import sys
!{sys.executable} -m ensurepip --upgrade
!{sys.executable} -m pip install --upgrade pip setuptools wheel
!{sys.executable} -m pip install numpy pandas matplotlib scikit-learn keras xgboost

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import cross_val_score, GridSearchCV
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
import pickle

ROOT = Path("kaggle-dataset")
TEST_ROOT = Path("new_test")

In [ ]:
ax_t1 = pd.read_csv(ROOT / "radiomics_info" / "train" / "ax_t1_radiomics_train.csv")
ax_t1c = pd.read_csv(ROOT / "radiomics_info" / "train" / "ax_t1c_radiomics_train.csv")
ax_t2 = pd.read_csv(ROOT / "radiomics_info" / "train" / "ax_t2_radiomics_train.csv")
ax_t2f = pd.read_csv(ROOT / "radiomics_info" / "train" / "ax_t2f_radiomics_train.csv")
    

In [ ]:
# Rename radiomics columns to include modality as prefix
dfs = [ax_t1, ax_t1c, ax_t2, ax_t2f]
modality_names = ['ax_t1', 'ax_t1c', 'ax_t2', 'ax_t2f']

dfs_renamed = []
for df, modality in zip(dfs, modality_names):
    df_copy = df.drop(columns=['sex', 'age', 'modality']).copy()
    # Rename all radiomics columns to have modality prefix
    radiomics_cols = [col for col in df_copy.columns if col != 'case_id']
    new_names = {col: f'{modality}_{col}' for col in radiomics_cols}
    df_copy = df_copy.rename(columns=new_names)
    dfs_renamed.append(df_copy)

# Merge all on case_id
final_df = dfs_renamed[0]
for df in dfs_renamed[1:]:
    final_df = pd.merge(final_df, df, on='case_id')

final_df

In [ ]:
clinical_total = pd.read_csv(ROOT / "clinical_information" / "train_patient_info.csv")
clinical_total

### Note to self:
#### Clinical information has radiomics data as QUALITATIVE variables
#### Radiomic information has radiomic data as QUANTITATIVE variables, but more missing data (e.g artifact loss)

In [ ]:
clinical = clinical_total[["case_id", "Sex", "Age"]]
clinical

### Skewed data, use median

In [ ]:
ages = clinical.Age.fillna(clinical.Age.median())
ages

In [ ]:
clinical

In [ ]:
import matplotlib.pyplot as plt
age2 = clinical.Age
age2.dropna()
plt.hist(age2)

In [ ]:
clinical_encoded = pd.get_dummies(data=clinical, columns=["Sex"])
clinical_encoded

In [ ]:
clinical_encoded = pd.get_dummies(data=clinical, columns=["Sex"])
clinical_encoded

### Train features

In [ ]:
final_features = pd.merge(clinical_encoded, final_df, on="case_id")
final_features = final_features.drop(columns=["case_id"])

In [ ]:
final_features

### Train class

In [ ]:
train = pd.read_json(ROOT / "train.json")
train_class = train.T.Overall_class
train_class = pd.DataFrame(train_class, columns=['Overall_class'])
train_class

### Validation set

In [ ]:
valid = pd.read_json(ROOT / "val.json")
valid_class = valid.T.Overall_class
valid_class = pd.DataFrame(valid_class, columns=['Overall_class'])
valid_class

In [ ]:
ax_t1 = pd.read_csv(ROOT / "radiomics_info" / "val" / "ax_t1_radiomics_val.csv")
ax_t1c = pd.read_csv(ROOT / "radiomics_info" / "val" / "ax_t1c_radiomics_val.csv")
ax_t2 = pd.read_csv(ROOT / "radiomics_info" / "val" / "ax_t2_radiomics_val.csv")
ax_t2f = pd.read_csv(ROOT / "radiomics_info" / "val" / "ax_t2f_radiomics_val.csv")

dfs_val = [ax_t1, ax_t1c, ax_t2, ax_t2f]
modality_names = ['ax_t1', 'ax_t1c', 'ax_t2', 'ax_t2f']

dfs_val_renamed = []
for df_val, modality in zip(dfs_val, modality_names):  # Fixed: dfs2 -> dfs_val
    df_val_copy = df_val.drop(columns=['sex', 'age', 'modality']).copy()
    # Rename all radiomics columns to have modality prefix
    radiomics_cols = [col for col in df_val_copy.columns if col != 'case_id']
    new_names = {col: f'{modality}_{col}' for col in radiomics_cols}
    df_val_copy = df_val_copy.rename(columns=new_names)
    dfs_val_renamed.append(df_val_copy)  # Fixed: df2_val -> df_val_copy

# Merge all on case_id
final_df_val = dfs_val_renamed[0]
for df in dfs_val_renamed[1:]:
    final_df_val = pd.merge(final_df_val, df, on='case_id')

final_df_val 

In [ ]:
clinical_test_total = pd.read_csv(ROOT / "clinical_information" / "val_patient_info.csv")
clinical_test_total

In [ ]:
clinical_val = clinical_test_total[["case_id", "Sex", "Age"]]
clinical_val_encoded = pd.get_dummies(clinical_val, columns=['Sex'])
clinical_val_encoded

In [ ]:
final_val_features = pd.merge(clinical_val_encoded, final_df_val, on="case_id")
final_val_features = final_val_features.drop(columns=["case_id"])
final_val_features

### Tune

In [ ]:
from sklearn.preprocessing import LabelEncoder
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score

le = LabelEncoder()
train_class_enc = le.fit_transform(train_class['Overall_class'])
valid_class_enc = le.transform(valid_class['Overall_class'])

model = XGBClassifier(
    objective='multi:softprob',
    num_class=len(le.classes_),
    max_depth=4,
    learning_rate=0.3,
    n_estimators=100,
    use_label_encoder=False,
    random_state=42
)
model.fit(final_features, train_class_enc)

y_pred = model.predict(final_val_features)
val_acc = accuracy_score(valid_class_enc, y_pred)
print(val_acc)

In [ ]:
from sklearn.preprocessing import LabelEncoder
from xgboost import XGBClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import accuracy_score

le = LabelEncoder()
train_class_enc = le.fit_transform(train_class['Overall_class'])
valid_class_enc = le.transform(valid_class['Overall_class'])

param_grid = {
    'max_depth': [3, 4, 5, 6],
    'learning_rate': [0.01, 0.1, 0.3, 0.5],
    'n_estimators': [50, 100, 150, 200],
    'subsample': [0.6, 0.8, 1.0],
    'colsample_bytree': [0.6, 0.8, 1.0],
    'min_child_weight': [1, 3, 5]
}

model = XGBClassifier(
    objective='multi:softprob',
    num_class=len(le.classes_),
    random_state=42,
    eval_metric='mlogloss'  
)

grid_search = GridSearchCV(
    estimator=model,
    param_grid=param_grid,
    cv=5,  
    scoring='accuracy',
    n_jobs=-1, 
    verbose=1
)

grid_search.fit(final_features, train_class_enc)

print("Best parameters:", grid_search.best_params_)
print("Best cross-validation score: {:.4f}".format(grid_search.best_score_))

best_model = grid_search.best_estimator_
y_pred = best_model.predict(final_val_features)
val_acc = accuracy_score(valid_class_enc, y_pred)
print(f"Validation Accuracy: {val_acc:.4f}")

### Test Features

In [ ]:
ax_t1 = pd.read_csv(TEST_ROOT / "radiomics_info" / "test" / "ax_t1_radiomics_test.csv")
ax_t1c = pd.read_csv(TEST_ROOT / "radiomics_info" / "test" / "ax_t1c_radiomics_test.csv")
ax_t2 = pd.read_csv(TEST_ROOT / "radiomics_info" / "test" / "ax_t2_radiomics_test.csv")
ax_t2f = pd.read_csv(TEST_ROOT / "radiomics_info" / "test" / "ax_t2f_radiomics_test.csv")

dfs2 = [ax_t1, ax_t1c, ax_t2, ax_t2f]
modality_names = ['ax_t1', 'ax_t1c', 'ax_t2', 'ax_t2f']

dfs2_renamed = []
for df2, modality in zip(dfs2, modality_names):
    drop_cols = [c for c in ['sex', 'age', 'modality', 'Sex', 'Age', 'Modality'] if c in df2.columns]
    df2_copy = df2.drop(columns=drop_cols).copy()
    # Rename all radiomics columns to have modality prefix
    radiomics_cols = [col for col in df2_copy.columns if col != 'case_id']
    new_names = {col: f'{modality}_{col}' for col in radiomics_cols}
    df2_copy = df2_copy.rename(columns=new_names)
    dfs2_renamed.append(df2_copy)

# Merge all on case_id
final_df2 = dfs2_renamed[0]
for df in dfs2_renamed[1:]:
    final_df2 = pd.merge(final_df2, df, on='case_id')

final_df2


In [ ]:
clinical_test_total = pd.read_csv(TEST_ROOT / "clinical_information" / "test" / "test_patient_info.csv")
clinical_test_total

In [ ]:
clinical_test = clinical_test_total[["case_id", "Sex", "Age"]]
clinical_test_encoded = pd.get_dummies(clinical_test, columns=['Sex'])
clinical_test_encoded

In [ ]:
final_test_features = pd.merge(clinical_test_encoded, final_df2, on="case_id")
test_case_id = final_test_features["case_id"].copy()
final_test_features = final_test_features.drop(columns=["case_id"])

In [ ]:
final_test_features

In [ ]:
# 0.967
from sklearn.preprocessing import LabelEncoder
from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score


le = LabelEncoder()
train_class_enc = le.fit_transform(train_class['Overall_class']) 
model = XGBClassifier(objective='multi:softprob', num_class=len(le.classes_), max_depth=4, learning_rate=0.3, n_estimators=100, use_label_encoder=False, random_state=42)
model.fit(final_features, train_class_enc)

y_pred = model.predict(final_test_features)


In [ ]:
from sklearn.preprocessing import LabelEncoder
from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score


le = LabelEncoder()
train_class_enc = le.fit_transform(train_class['Overall_class']) 
model = XGBClassifier(objective='multi:softprob', num_class=len(le.classes_), max_depth=6, learning_rate=0.1, n_estimators=200, colsample_bytree=1, min_child_weight=1, use_label_encoder=False, random_state=42)
model.fit(final_features, train_class_enc)

y_pred = model.predict(final_test_features)

In [ ]:
y_pred_labels = le.inverse_transform(y_pred)

In [ ]:
y_pred_labels

In [ ]:
case_id = pd.Series(test_case_id)
predictions = pd.Series(y_pred_labels)
merged = [case_id, predictions]
submission = pd.concat(merged, axis=1)
submission.columns = ['case_id', 'Overall_class']
submission.to_csv(TEST_ROOT / 'sample_submission_xgb.csv', index=False)

In [ ]:
predictions = pd.Series(y_pred_labels)

In [ ]:
# Quick diagnostics for feature alignment and prediction balance
train_cols = list(final_features.columns)
val_cols = list(final_val_features.columns)
test_cols = list(final_test_features.columns)

print('train shape:', final_features.shape)
print('val shape:', final_val_features.shape)
print('test shape:', final_test_features.shape)

print('has case_id in train/val/test:', 'case_id' in train_cols, 'case_id' in val_cols, 'case_id' in test_cols)

train_set = set(train_cols)
val_set = set(val_cols)
test_set = set(test_cols)

print('val missing vs train:', len(train_set - val_set))
print('val extra vs train:', len(val_set - train_set))
print('test missing vs train:', len(train_set - test_set))
print('test extra vs train:', len(test_set - train_set))

if len(train_set - test_set) or len(test_set - train_set):
    print('sample missing test cols:', sorted(list(train_set - test_set))[:10])
    print('sample extra test cols:', sorted(list(test_set - train_set))[:10])

if 'y_pred_labels' in globals():
    print('prediction distribution:')
    print(pd.Series(y_pred_labels).value_counts(normalize=True).sort_values(ascending=False).head(10))

In [ ]:
# Strategy v2: stratified CV + compact randomized search + train+val refit
from sklearn.model_selection import StratifiedKFold, RandomizedSearchCV
from xgboost import XGBClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score
import numpy as np

X_train = final_features.copy()
X_val = final_val_features.copy()
X_test = final_test_features.copy()

y_train = train_class['Overall_class'].copy()
y_val = valid_class['Overall_class'].copy()

le_v2 = LabelEncoder()
y_train_enc = le_v2.fit_transform(y_train)
y_val_enc = le_v2.transform(y_val)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

param_dist = {
    'max_depth': [3, 4, 5, 6],
    'learning_rate': [0.03, 0.05, 0.08, 0.1],
    'n_estimators': [300, 500, 700, 900],
    'subsample': [0.7, 0.8, 0.9, 1.0],
    'colsample_bytree': [0.7, 0.8, 0.9, 1.0],
    'min_child_weight': [1, 3, 5, 7],
    'gamma': [0, 0.1, 0.3],
    'reg_alpha': [0, 0.05, 0.1, 0.5],
    'reg_lambda': [1, 2, 5]
}

base_model = XGBClassifier(
    objective='multi:softprob',
    num_class=len(le_v2.classes_),
    eval_metric='mlogloss',
    tree_method='hist',
    random_state=42
)

search_v2 = RandomizedSearchCV(
    estimator=base_model,
    param_distributions=param_dist,
    n_iter=24,
    scoring='accuracy',
    cv=cv,
    random_state=42,
    n_jobs=-1,
    verbose=1
)

search_v2.fit(X_train, y_train_enc)

best_params_v2 = search_v2.best_params_
best_cv_v2 = search_v2.best_score_
print('Best CV acc:', round(best_cv_v2, 4))
print('Best params:', best_params_v2)

best_model_v2 = search_v2.best_estimator_
val_pred_v2 = best_model_v2.predict(X_val)
val_acc_v2 = accuracy_score(y_val_enc, val_pred_v2)
print('Validation acc (sanity):', round(val_acc_v2, 4))

In [ ]:
# Refit with all labeled data (train + val), then export submission
X_all = pd.concat([X_train, X_val], axis=0, ignore_index=True)
y_all = pd.concat([y_train, y_val], axis=0, ignore_index=True)

le_all = LabelEncoder()
y_all_enc = le_all.fit_transform(y_all)

params_all = best_params_v2.copy()
model_all = XGBClassifier(
    objective='multi:softprob',
    num_class=len(le_all.classes_),
    eval_metric='mlogloss',
    tree_method='hist',
    random_state=42,
    **params_all
)
model_all.fit(X_all, y_all_enc)

test_pred_all = model_all.predict(X_test)
test_labels_all = le_all.inverse_transform(test_pred_all)

submission_v2 = pd.DataFrame({
    'case_id': test_case_id.values,
    'Overall_class': test_labels_all
})
submission_v2.to_csv(TEST_ROOT / 'sample_submission_xgb.csv', index=False)
submission_v2.head()

In [ ]:
# Strategy v3: seed ensemble (validate first)
from sklearn.metrics import accuracy_score
from xgboost import XGBClassifier
import numpy as np

# Use tuned params from v2 as base, plus one slightly more regularized variant
params_base = best_params_v2.copy()
params_alt = best_params_v2.copy()
params_alt['max_depth'] = max(3, params_alt['max_depth'] - 1)
params_alt['min_child_weight'] = max(3, params_alt['min_child_weight'])
params_alt['reg_alpha'] = max(0.1, params_alt['reg_alpha'])

param_list = [params_base, params_alt]
seed_list = [42, 2024, 3407]

val_prob_ens = np.zeros((X_val.shape[0], len(le_v2.classes_)), dtype=float)
n_models = 0

for params in param_list:
    for seed in seed_list:
        model_ens = XGBClassifier(
            objective='multi:softprob',
            num_class=len(le_v2.classes_),
            eval_metric='mlogloss',
            tree_method='hist',
            random_state=seed,
            **params
        )
        model_ens.fit(X_train, y_train_enc)
        val_prob_ens += model_ens.predict_proba(X_val)
        n_models += 1

val_prob_ens /= n_models
val_pred_ens = np.argmax(val_prob_ens, axis=1)
val_acc_ens = accuracy_score(y_val_enc, val_pred_ens)

print('v2 single-model val acc:', round(val_acc_v2, 4))
print('v3 ensemble val acc:', round(val_acc_ens, 4))
print('ensemble models:', n_models)

In [ ]:
# If v3 validation is not worse, refit ensemble on all labels and export
use_ens = val_acc_ens >= val_acc_v2
print('use ensemble for submission:', use_ens)

test_prob_final = np.zeros((X_test.shape[0], len(le_all.classes_)), dtype=float)
n_models_all = 0

if use_ens:
    X_all = pd.concat([X_train, X_val], axis=0, ignore_index=True)
    y_all = pd.concat([y_train, y_val], axis=0, ignore_index=True)

    le_all = LabelEncoder()
    y_all_enc = le_all.fit_transform(y_all)

    for params in param_list:
        for seed in seed_list:
            model_all_ens = XGBClassifier(
                objective='multi:softprob',
                num_class=len(le_all.classes_),
                eval_metric='mlogloss',
                tree_method='hist',
                random_state=seed,
                **params
            )
            model_all_ens.fit(X_all, y_all_enc)
            test_prob_final += model_all_ens.predict_proba(X_test)
            n_models_all += 1

    test_prob_final /= n_models_all
    test_pred_final = np.argmax(test_prob_final, axis=1)
    test_labels_final = le_all.inverse_transform(test_pred_final)
else:
    # fallback to existing v2 single-model submission labels
    test_labels_final = test_labels_all

submission_v3 = pd.DataFrame({
    'case_id': test_case_id.values,
    'Overall_class': test_labels_final
})
submission_v3.to_csv(TEST_ROOT / 'sample_submission_xgb.csv', index=False)
submission_v3.to_csv(TEST_ROOT / 'sample_submission_xgb_v3.csv', index=False)
submission_v3.head()

In [ ]:
# Optimized multimodal pipeline - data ingestion + feature engineering
from pathlib import Path
import json
import numpy as np
import pandas as pd

from sklearn.decomposition import PCA, TruncatedSVD
from sklearn.feature_extraction.text import TfidfVectorizer

ROOT = Path('kaggle-dataset')
TEST_ROOT = Path('new_test')
MODALITIES = ['ax_t1', 'ax_t1c', 'ax_t2', 'ax_t2f']

def _json_to_label_df(json_path):
    d = pd.read_json(json_path).T.reset_index().rename(columns={'index': 'case_id'})
    d['case_id'] = d['case_id'].astype(str)
    if 'Overall_class' in d.columns:
        return d[['case_id', 'Overall_class']]
    return d[['case_id']]

def _read_clinical(root, split):
    if split == 'train':
        p = root / 'clinical_information' / 'train_patient_info.csv'
    elif split == 'val':
        p = root / 'clinical_information' / 'val_patient_info.csv'
    else:
        p = root / 'clinical_information' / 'test' / 'test_patient_info.csv'
    df = pd.read_csv(p)
    df['case_id'] = df['case_id'].astype(str)
    return df

def _read_reports(root, split):
    if split == 'train':
        p = root / 'original_raw_report' / 'train_patient_info.csv'
    elif split == 'val':
        p = root / 'original_raw_report' / 'val_patient_info.csv'
    else:
        p = root / 'original_raw_report' / 'test_patient_info.csv'
    df = pd.read_csv(p)
    df['case_id'] = df['case_id'].astype(str)
    if 'raw_report' not in df.columns:
        text_cols = [c for c in df.columns if c != 'case_id']
        df['raw_report'] = df[text_cols].astype(str).agg(' '.join, axis=1) if text_cols else ''
    return df[['case_id', 'raw_report']]

def _read_radiomics(root, split):
    frames = []
    for m in MODALITIES:
        p = root / 'radiomics_info' / split / f'{m}_radiomics_{split}.csv'
        d = pd.read_csv(p)
        d['case_id'] = d['case_id'].astype(str)
        drop_cols = [c for c in ['sex', 'age', 'modality', 'Sex', 'Age', 'Modality'] if c in d.columns]
        d = d.drop(columns=drop_cols)
        rename_cols = {c: f'{m}_{c}' for c in d.columns if c != 'case_id'}
        d = d.rename(columns=rename_cols)
        frames.append(d)
    out = frames[0]
    for f in frames[1:]:
        out = out.merge(f, on='case_id', how='outer')
    return out

def _load_deep_vectors(root, case_ids):
    feat_root = root / 'image_features' / 'image_features'
    rows = []
    for cid in case_ids:
        per_case = []
        for m in MODALITIES:
            p = feat_root / str(cid) / m / 'image.npy'
            if p.exists():
                v = np.load(p).astype(np.float32).ravel()
            else:
                v = np.full((2048,), np.nan, dtype=np.float32)
            per_case.append(v)
        vec = np.concatenate(per_case, axis=0)
        rows.append((str(cid), vec))
    case_col = [r[0] for r in rows]
    mat = np.stack([r[1] for r in rows], axis=0) if rows else np.empty((0, 8192), dtype=np.float32)
    return case_col, mat

def build_split_frames():
    y_train = _json_to_label_df(ROOT / 'train.json')
    y_val = _json_to_label_df(ROOT / 'val.json')
    test_ids = _json_to_label_df(TEST_ROOT / 'test.json')

    train_base = y_train[['case_id']].merge(_read_clinical(ROOT, 'train'), on='case_id', how='left')
    val_base = y_val[['case_id']].merge(_read_clinical(ROOT, 'val'), on='case_id', how='left')
    test_base = test_ids[['case_id']].merge(_read_clinical(TEST_ROOT, 'test'), on='case_id', how='left')

    train_base = train_base.merge(_read_reports(ROOT, 'train'), on='case_id', how='left')
    val_base = val_base.merge(_read_reports(ROOT, 'val'), on='case_id', how='left')
    test_base = test_base.merge(_read_reports(TEST_ROOT, 'test'), on='case_id', how='left')

    train_rad = _read_radiomics(ROOT, 'train')
    val_rad = _read_radiomics(ROOT, 'val')
    test_rad = _read_radiomics(TEST_ROOT, 'test')

    train_df = train_base.merge(train_rad, on='case_id', how='left')
    val_df = val_base.merge(val_rad, on='case_id', how='left')
    test_df = test_base.merge(test_rad, on='case_id', how='left')

    return train_df, val_df, test_df, y_train, y_val

train_df_raw, val_df_raw, test_df_raw, y_train_df, y_val_df = build_split_frames()
print('raw shapes:', train_df_raw.shape, val_df_raw.shape, test_df_raw.shape)

# Clinical categorical handling (keep NaN as NaN)
cat_cols = [
    'Sex',
    'Tumor Location',
    'Signal Intensity (T1)',
    'Signal Intensity (T1c)',
    'Signal Intensity (T2)',
    'Signal Intensity (T2-FLAIR)'
]
for c in cat_cols:
    if c in train_df_raw.columns:
        train_df_raw[c] = train_df_raw[c].astype('category')
    if c in val_df_raw.columns:
        val_df_raw[c] = val_df_raw[c].astype('category')
    if c in test_df_raw.columns:
        test_df_raw[c] = test_df_raw[c].astype('category')

# Radiomics feature filtering on TRAIN only
radiomics_cols = [c for c in train_df_raw.columns if 'rad_' in c]
rad_train = train_df_raw[radiomics_cols]

# Drop zero-variance columns (all equal including NaN pattern)
nunique = rad_train.nunique(dropna=False)
keep_rad_1 = nunique[nunique > 1].index.tolist()

# Drop near-duplicate columns by abs correlation > 0.99 (pairwise on available data)
corr = train_df_raw[keep_rad_1].corr(numeric_only=True).abs()
upper = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))
drop_corr = [col for col in upper.columns if any(upper[col] > 0.99)]
keep_rad = [c for c in keep_rad_1 if c not in drop_corr]
print('radiomics kept:', len(keep_rad), 'dropped:', len(radiomics_cols) - len(keep_rad))

# Deep image features -> PCA
train_ids = train_df_raw['case_id'].tolist()
val_ids = val_df_raw['case_id'].tolist()
test_ids = test_df_raw['case_id'].tolist()

_, deep_train = _load_deep_vectors(ROOT, train_ids)
_, deep_val = _load_deep_vectors(ROOT, val_ids)
_, deep_test = _load_deep_vectors(TEST_ROOT, test_ids)

# Keep NaN policy overall, but PCA cannot fit NaN => fill deep only with train medians
deep_median = np.nanmedian(deep_train, axis=0)
deep_median = np.where(np.isnan(deep_median), 0.0, deep_median)
deep_train_f = np.where(np.isnan(deep_train), deep_median, deep_train)
deep_val_f = np.where(np.isnan(deep_val), deep_median, deep_val)
deep_test_f = np.where(np.isnan(deep_test), deep_median, deep_test)

deep_pca = PCA(n_components=32, random_state=42)
deep_train_comp = deep_pca.fit_transform(deep_train_f)
deep_val_comp = deep_pca.transform(deep_val_f)
deep_test_comp = deep_pca.transform(deep_test_f)

deep_cols = [f'deep_pca_{i:02d}' for i in range(deep_train_comp.shape[1])]
deep_train_df = pd.DataFrame(deep_train_comp, columns=deep_cols)
deep_val_df = pd.DataFrame(deep_val_comp, columns=deep_cols)
deep_test_df = pd.DataFrame(deep_test_comp, columns=deep_cols)

# Raw report NLP -> TFIDF + SVD
train_text = train_df_raw['raw_report'].fillna('').astype(str).tolist()
val_text = val_df_raw['raw_report'].fillna('').astype(str).tolist()
test_text = test_df_raw['raw_report'].fillna('').astype(str).tolist()

tfidf = TfidfVectorizer(max_features=900, stop_words='english', ngram_range=(1, 2))
Xtr_tfidf = tfidf.fit_transform(train_text)
Xva_tfidf = tfidf.transform(val_text)
Xte_tfidf = tfidf.transform(test_text)

svd = TruncatedSVD(n_components=30, random_state=42)
train_txt_comp = svd.fit_transform(Xtr_tfidf)
val_txt_comp = svd.transform(Xva_tfidf)
test_txt_comp = svd.transform(Xte_tfidf)

txt_cols = [f'txt_svd_{i:02d}' for i in range(train_txt_comp.shape[1])]
txt_train_df = pd.DataFrame(train_txt_comp, columns=txt_cols)
txt_val_df = pd.DataFrame(val_txt_comp, columns=txt_cols)
txt_test_df = pd.DataFrame(test_txt_comp, columns=txt_cols)

# Build final tabular matrices (leave missing values as NaN)
base_cols = ['case_id', 'Age'] + [c for c in cat_cols if c in train_df_raw.columns]
train_tab = train_df_raw[base_cols + keep_rad].reset_index(drop=True)
val_tab = val_df_raw[base_cols + keep_rad].reset_index(drop=True)
test_tab = test_df_raw[base_cols + keep_rad].reset_index(drop=True)

X_train_tab = pd.concat([train_tab, deep_train_df, txt_train_df], axis=1)
X_val_tab = pd.concat([val_tab, deep_val_df, txt_val_df], axis=1)
X_test_tab = pd.concat([test_tab, deep_test_df, txt_test_df], axis=1)

print('tabular shapes:', X_train_tab.shape, X_val_tab.shape, X_test_tab.shape)

In [ ]:
# Optimized training - stratified CV + class-balance weights + threshold tuning
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import f1_score
from sklearn.utils.class_weight import compute_class_weight
from xgboost import XGBClassifier
import numpy as np
import pandas as pd

# Keep case_id for submission only
train_case_id = X_train_tab['case_id'].copy()
val_case_id = X_val_tab['case_id'].copy()
test_case_id_opt = X_test_tab['case_id'].copy()

X_train_model = X_train_tab.drop(columns=['case_id'])
X_val_model = X_val_tab.drop(columns=['case_id'])
X_test_model = X_test_tab.drop(columns=['case_id'])

# XGBoost categorical support requires category dtype in dataframe
for c in X_train_model.columns:
    if str(X_train_model[c].dtype) == 'category':
        X_val_model[c] = X_val_model[c].astype('category')
        X_test_model[c] = X_test_model[c].astype('category')

le_opt = LabelEncoder()
y_train = le_opt.fit_transform(y_train_df['Overall_class'])
y_val = le_opt.transform(y_val_df['Overall_class'])

classes = np.unique(y_train)
class_weights = compute_class_weight(class_weight='balanced', classes=classes, y=y_train)
cw_map = {c: w for c, w in zip(classes, class_weights)}
sample_weight_train = np.array([cw_map[c] for c in y_train], dtype=float)

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
oof_proba = np.zeros((X_train_model.shape[0], len(classes)), dtype=float)
best_iters = []

base_params = dict(
    objective='multi:softprob',
    num_class=len(classes),
    eval_metric='mlogloss',
    tree_method='hist',
    learning_rate=0.05,
    max_depth=6,
    min_child_weight=3,
    subsample=0.85,
    colsample_bytree=0.9,
    n_estimators=2000,
    reg_lambda=5,
    reg_alpha=0.1,
    gamma=0.0,
    random_state=42,
    enable_categorical=True
)

for fold, (tr_idx, va_idx) in enumerate(skf.split(X_train_model, y_train), start=1):
    X_tr, X_va = X_train_model.iloc[tr_idx], X_train_model.iloc[va_idx]
    y_tr, y_va = y_train[tr_idx], y_train[va_idx]
    w_tr = sample_weight_train[tr_idx]

    clf = XGBClassifier(**base_params)
    clf.fit(
        X_tr, y_tr,
        sample_weight=w_tr,
        eval_set=[(X_va, y_va)],
        verbose=False
    )

    if hasattr(clf, 'best_iteration') and clf.best_iteration is not None:
        best_iters.append(int(clf.best_iteration))
    else:
        best_iters.append(base_params['n_estimators'])

    oof_proba[va_idx] = clf.predict_proba(X_va)
    oof_pred = np.argmax(oof_proba[va_idx], axis=1)
    fold_f1 = f1_score(y_va, oof_pred, average='macro')
    print(f'fold {fold} macro_f1: {fold_f1:.4f}')

base_oof_pred = np.argmax(oof_proba, axis=1)
base_oof_f1 = f1_score(y_train, base_oof_pred, average='macro')
print('OOF macro_f1 (argmax):', round(base_oof_f1, 4))

# Threshold (class scale) optimization on OOF probs for macro-F1
scales = np.ones(len(classes), dtype=float)
grid = [0.7, 0.85, 1.0, 1.15, 1.3, 1.5]

def scaled_predict(prob, s):
    return np.argmax(prob * s.reshape(1, -1), axis=1)

best_f1 = base_oof_f1
for k in range(len(classes)):
    best_s = scales[k]
    for g in grid:
        cand = scales.copy()
        cand[k] = g
        pred = scaled_predict(oof_proba, cand)
        f1 = f1_score(y_train, pred, average='macro')
        if f1 > best_f1:
            best_f1 = f1
            best_s = g
    scales[k] = best_s

print('OOF macro_f1 (scaled):', round(best_f1, 4))
print('class scales:', {le_opt.classes_[i]: float(scales[i]) for i in range(len(scales))})

# Train final model on train+val with weighted samples
X_all = pd.concat([X_train_model, X_val_model], axis=0, ignore_index=True)
y_all_labels = pd.concat([y_train_df['Overall_class'], y_val_df['Overall_class']], axis=0, ignore_index=True)
y_all = le_opt.transform(y_all_labels)
sample_weight_all = np.array([cw_map[c] for c in y_all], dtype=float)

final_estimators = int(np.clip(np.mean(best_iters) * 1.15, 300, 2000))
final_params = base_params.copy()
final_params['n_estimators'] = final_estimators

final_clf = XGBClassifier(**final_params)
final_clf.fit(X_all, y_all, sample_weight=sample_weight_all, verbose=False)

test_proba = final_clf.predict_proba(X_test_model)
test_pred_scaled = scaled_predict(test_proba, scales)
test_pred_labels = le_opt.inverse_transform(test_pred_scaled)

submission_opt = pd.DataFrame({
    'case_id': test_case_id_opt.values,
    'Overall_class': test_pred_labels
})
submission_opt.to_csv(TEST_ROOT / 'sample_submission_xgb_optimized.csv', index=False)
submission_opt.to_csv(TEST_ROOT / 'sample_submission_xgb.csv', index=False)
submission_opt.head()

In [ ]:
# Rebuild optimized features with robust test report path fallback
def _read_reports(root, split):
    if split == 'train':
        candidates = [root / 'original_raw_report' / 'train_patient_info.csv']
    elif split == 'val':
        candidates = [root / 'original_raw_report' / 'val_patient_info.csv']
    else:
        candidates = [
            root / 'original_raw_report' / 'test' / 'test_patient_info.csv',
            root / 'original_raw_report' / 'test_patient_info.csv',
        ]
    p = next((x for x in candidates if x.exists()), None)
    if p is None:
        raise FileNotFoundError(f'No raw report file found for split={split}: {candidates}')
    df = pd.read_csv(p)
    df['case_id'] = df['case_id'].astype(str)
    if 'raw_report' not in df.columns:
        text_cols = [c for c in df.columns if c != 'case_id']
        df['raw_report'] = df[text_cols].astype(str).agg(' '.join, axis=1) if text_cols else ''
    return df[['case_id', 'raw_report']]

train_df_raw, val_df_raw, test_df_raw, y_train_df, y_val_df = build_split_frames()
print('raw shapes:', train_df_raw.shape, val_df_raw.shape, test_df_raw.shape)

cat_cols = [
    'Sex',
    'Tumor Location',
    'Signal Intensity (T1)',
    'Signal Intensity (T1c)',
    'Signal Intensity (T2)',
    'Signal Intensity (T2-FLAIR)'
]
for c in cat_cols:
    if c in train_df_raw.columns:
        train_df_raw[c] = train_df_raw[c].astype('category')
    if c in val_df_raw.columns:
        val_df_raw[c] = val_df_raw[c].astype('category')
    if c in test_df_raw.columns:
        test_df_raw[c] = test_df_raw[c].astype('category')

radiomics_cols = [c for c in train_df_raw.columns if 'rad_' in c]
rad_train = train_df_raw[radiomics_cols]
nunique = rad_train.nunique(dropna=False)
keep_rad_1 = nunique[nunique > 1].index.tolist()
corr = train_df_raw[keep_rad_1].corr(numeric_only=True).abs()
upper = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))
drop_corr = [col for col in upper.columns if any(upper[col] > 0.99)]
keep_rad = [c for c in keep_rad_1 if c not in drop_corr]
print('radiomics kept:', len(keep_rad), 'dropped:', len(radiomics_cols) - len(keep_rad))

train_ids = train_df_raw['case_id'].tolist()
val_ids = val_df_raw['case_id'].tolist()
test_ids = test_df_raw['case_id'].tolist()

_, deep_train = _load_deep_vectors(ROOT, train_ids)
_, deep_val = _load_deep_vectors(ROOT, val_ids)
_, deep_test = _load_deep_vectors(TEST_ROOT, test_ids)

deep_median = np.nanmedian(deep_train, axis=0)
deep_median = np.where(np.isnan(deep_median), 0.0, deep_median)
deep_train_f = np.where(np.isnan(deep_train), deep_median, deep_train)
deep_val_f = np.where(np.isnan(deep_val), deep_median, deep_val)
deep_test_f = np.where(np.isnan(deep_test), deep_median, deep_test)

deep_pca = PCA(n_components=32, random_state=42)
deep_train_comp = deep_pca.fit_transform(deep_train_f)
deep_val_comp = deep_pca.transform(deep_val_f)
deep_test_comp = deep_pca.transform(deep_test_f)
deep_cols = [f'deep_pca_{i:02d}' for i in range(deep_train_comp.shape[1])]
deep_train_df = pd.DataFrame(deep_train_comp, columns=deep_cols)
deep_val_df = pd.DataFrame(deep_val_comp, columns=deep_cols)
deep_test_df = pd.DataFrame(deep_test_comp, columns=deep_cols)

train_text = train_df_raw['raw_report'].fillna('').astype(str).tolist()
val_text = val_df_raw['raw_report'].fillna('').astype(str).tolist()
test_text = test_df_raw['raw_report'].fillna('').astype(str).tolist()
tfidf = TfidfVectorizer(max_features=900, stop_words='english', ngram_range=(1, 2))
Xtr_tfidf = tfidf.fit_transform(train_text)
Xva_tfidf = tfidf.transform(val_text)
Xte_tfidf = tfidf.transform(test_text)
svd = TruncatedSVD(n_components=30, random_state=42)
train_txt_comp = svd.fit_transform(Xtr_tfidf)
val_txt_comp = svd.transform(Xva_tfidf)
test_txt_comp = svd.transform(Xte_tfidf)
txt_cols = [f'txt_svd_{i:02d}' for i in range(train_txt_comp.shape[1])]
txt_train_df = pd.DataFrame(train_txt_comp, columns=txt_cols)
txt_val_df = pd.DataFrame(val_txt_comp, columns=txt_cols)
txt_test_df = pd.DataFrame(test_txt_comp, columns=txt_cols)

base_cols = ['case_id', 'Age'] + [c for c in cat_cols if c in train_df_raw.columns]
train_tab = train_df_raw[base_cols + keep_rad].reset_index(drop=True)
val_tab = val_df_raw[base_cols + keep_rad].reset_index(drop=True)
test_tab = test_df_raw[base_cols + keep_rad].reset_index(drop=True)
X_train_tab = pd.concat([train_tab, deep_train_df, txt_train_df], axis=1)
X_val_tab = pd.concat([val_tab, deep_val_df, txt_val_df], axis=1)
X_test_tab = pd.concat([test_tab, deep_test_df, txt_test_df], axis=1)
print('tabular shapes:', X_train_tab.shape, X_val_tab.shape, X_test_tab.shape)

In [ ]:
# Final fit rescue: enforce categorical dtype after concat, then export
X_all = pd.concat([X_train_model, X_val_model], axis=0, ignore_index=True)
y_all_labels = pd.concat([y_train_df['Overall_class'], y_val_df['Overall_class']], axis=0, ignore_index=True)
y_all = le_opt.transform(y_all_labels)
sample_weight_all = np.array([cw_map[c] for c in y_all], dtype=float)

cat_like_cols = [
    'Sex',
    'Tumor Location',
    'Signal Intensity (T1)',
    'Signal Intensity (T1c)',
    'Signal Intensity (T2)',
    'Signal Intensity (T2-FLAIR)'
]
for c in cat_like_cols:
    if c in X_all.columns:
        X_all[c] = X_all[c].astype('category')
    if c in X_test_model.columns:
        X_test_model[c] = X_test_model[c].astype('category')

final_estimators = int(np.clip(np.mean(best_iters) * 1.15, 300, 2000))
final_params = base_params.copy()
final_params['n_estimators'] = final_estimators

final_clf = XGBClassifier(**final_params)
final_clf.fit(X_all, y_all, sample_weight=sample_weight_all, verbose=False)

def scaled_predict(prob, s):
    return np.argmax(prob * s.reshape(1, -1), axis=1)

test_proba = final_clf.predict_proba(X_test_model)
test_pred_scaled = scaled_predict(test_proba, scales)
test_pred_labels = le_opt.inverse_transform(test_pred_scaled)

submission_opt = pd.DataFrame({
    'case_id': test_case_id_opt.values,
    'Overall_class': test_pred_labels
})
submission_opt.to_csv(TEST_ROOT / 'sample_submission_xgb_optimized.csv', index=False)
submission_opt.to_csv(TEST_ROOT / 'sample_submission_xgb.csv', index=False)
submission_opt.head()